In [2]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import OpenAIEmbeddings

C:\Users\Lavanya Rajesh\AppData\Local\Temp\ipykernel_10892\107299174.py:3: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


STEP-1 : Load the pdf into text format

In [4]:
docs = PyPDFLoader("../2_RAG/Sample.pdf").load()

full_text = "\n".join([p.page_content for p in docs])
full_text

'All Across\nEUROPE\nEllen Weisberg and Ken Yoffe\nPublished by Ken Yoffe and Ellen Weisberg \nwww.facepaint.team\nAll Across Europe\nISBN: 978-1-64255- 778- 7\nLibrary of Congress Control Number: 2018933240\nCopyright © 2022\nAll rights reserved. No part of this book may be reproduced or transmitted in any form or by any means whatsoever \nwithout express written permission from the author , except in the case of brief quotations embodied in critical \narticles and reviews. Please refer all pertinent questions to the publisher. All rights reserved. No part of this book \nmay be reproduced or transmitted in any form or by any means, electronic or mechanical, including photocopying, \nrecording, or by an information storage and retrieval system except by a reviewer who may quote brief passages in a \nreview to be printed in a magazine or newspaper without permission in writing from the publisher.\n3\nSeeing Central and Southern Europe  \nwith Stevie  . . . . . . . . . . . . . . . . . . 

STEP-2 : Creating Chunks of the text data

In [5]:
chunker = SemanticChunker(
    OpenAIEmbeddings(model="text-embedding-3-small"),
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=60
)

semantic_chunks = chunker.create_documents([full_text])
semantic_chunks

[Document(metadata={}, page_content='All Across\nEUROPE\nEllen Weisberg and Ken Yoffe\nPublished by Ken Yoffe and Ellen Weisberg \nwww.facepaint.team\nAll Across Europe\nISBN: 978-1-64255- 778- 7\nLibrary of Congress Control Number: 2018933240\nCopyright © 2022\nAll rights reserved. No part of this book may be reproduced or transmitted in any form or by any means whatsoever \nwithout express written permission from the author , except in the case of brief quotations embodied in critical \narticles and reviews.'),
 Document(metadata={}, page_content='Please refer all pertinent questions to the publisher. All rights reserved.'),
 Document(metadata={}, page_content='No part of this book \nmay be reproduced or transmitted in any form or by any means, electronic or mechanical, including photocopying, \nrecording, or by an information storage and retrieval system except by a reviewer who may quote brief passages in a \nreview to be printed in a magazine or newspaper without permission in wri

Step-3&4 : Creating Embeddings and Storing them in Vector DB

In [6]:
from langchain_community.vectorstores import Chroma

In [7]:
embed_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [8]:
chroma_db = Chroma.from_documents(semantic_chunks, embed_model, persist_directory="./chroma_db_semantic")

Step-5 : Connection & Retrieval 

In [9]:
chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)

C:\Users\Lavanya Rajesh\AppData\Local\Temp\ipykernel_10892\2430328870.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)


In [10]:
chroma_db_con.similarity_search("tell me something about France?", k=3)

[Document(metadata={}, page_content='It  stretches  all the  way  from France \nand  reaches  the  U.K. Paris\n89\nThere is a high standard of living in France\nand a heavy emphasis on education . France\nand the U.K.'),
 Document(metadata={}, page_content='are linked by the Channel Tunnel,\nlocated underneath the English Channel . 88\nFRANCE\nFRANCE\n89\nParis  has  a lot  to  show  \nthe tourists  that  come  by: \nThe  Eiffel  Tower,  Louvre, \nand  the  Palace  of  Versailles.'),
 Document(metadata={}, page_content='are linked by the Channel Tunnel,\nlocated underneath the English Channel . 88\nFRANCE\n89\nParis  has  a lot  to  show  \nthe tourists  that  come  by: \nThe  Eiffel  Tower,  Louvre, \nand  the  Palace  of  Versailles.')]

Step-6 : LLM Integration and Answer Generation

In [11]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [12]:
user_query = input("Enter your question: ")

rel_chunks = chroma_db_con.similarity_search(user_query, k=3)

rel_chunks_content = []
for i, chunk in enumerate(rel_chunks):
    rel_chunks_content.append(chunk.page_content)
rel_chunks_content = str(rel_chunks_content)

llm.invoke(f"{user_query}, Use the following context to answer the question: {rel_chunks_content}")

AIMessage(content='France is a country known for its rich history, culture, and landmarks. Paris, the capital city, is a popular tourist destination with iconic attractions such as the Eiffel Tower, Louvre Museum, and Palace of Versailles. France is also connected to the United Kingdom by the Channel Tunnel, which runs underneath the English Channel. The country has a high standard of living and places a heavy emphasis on education. France is known for its culinary traditions, fashion, art, and architecture, making it a popular destination for travelers from around the world.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 110, 'prompt_tokens': 218, 'total_tokens': 328, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None,